# ARK-020 V4 — operator launcher (hardened)

Run cells in order. Cell 0 mounts Drive, force-refreshes the pinned executable, performs the read-only fail-closed resume scan, parses the machine marker, and **stops automatically** on an unsafe action. Cell 1 compiles and runs the full V4 test gate with live verbose output. Cell 2 starts/resumes the campaign and prints the real failure receipt if the child process fails.

In [ ]:
import json, subprocess, sys, os, shutil
from pathlib import Path

PINNED_RUNNER_COMMIT = 'f4244e2fea135cd768a5b5de5890e7117406c9c2'
REPO = '/content/An-Ra-the-new-AGI-ark020v4'
REPO_URL = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
SCAN_GATE_PASS = False
SAFE_ACTION = None

print('=== ARK-020 V4 CELL 0: MOUNT / PIN / SCAN ===')
print('STEP 1: mount Drive')
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=False)
    DRIVE_OK = Path('/content/drive/MyDrive').is_dir()
except Exception as exc:
    DRIVE_OK = False
    print('DRIVE MOUNT FAILED:', repr(exc))
if not DRIVE_OK:
    raise SystemExit('SAFE ACTION: STOP — DRIVE UNAVAILABLE')

print('STEP 2: create/refresh repository')
if os.path.exists(REPO) and not os.path.isdir(os.path.join(REPO, '.git')):
    print('Removing stale non-git path:', REPO)
    shutil.rmtree(REPO)
if not os.path.exists(REPO):
    subprocess.run(['git','clone','--depth','100','--branch','Arkenstone',REPO_URL,REPO], check=True)
else:
    subprocess.run(['git','-C',REPO,'remote','set-url','origin',REPO_URL], check=True)
    subprocess.run(['git','-C',REPO,'fetch','--depth','100','origin','Arkenstone'], check=True)

# Remove stale local edits/caches so a previous failed notebook run cannot poison this run.
subprocess.run(['git','-C',REPO,'reset','--hard'], check=True)
subprocess.run(['git','-C',REPO,'clean','-fd'], check=True)
subprocess.run(['git','-C',REPO,'checkout','--detach',PINNED_RUNNER_COMMIT], check=True)
head = subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'], text=True).strip()
assert head == PINNED_RUNNER_COMMIT, (head, PINNED_RUNNER_COMMIT)
print('PINNED SCIENTIFIC COMMIT OK:', head)

print('STEP 3: READ-ONLY resume scan')
runner = os.path.join(REPO, 'experiments/ARK-020-V4/run_ark020_v4.py')
scan = subprocess.run([sys.executable, runner, '--mode','scan','--drive-ok','True'],
                      cwd=REPO, capture_output=True, text=True)
print(scan.stdout)
if scan.stderr.strip():
    print('--- scan stderr ---')
    print(scan.stderr)
if scan.returncode != 0:
    raise SystemExit(f'SAFE ACTION: STOP — SCAN COMMAND FAILED (return code {scan.returncode})')
marker = '@@SCAN_JSON@@'
marker_line = next((ln for ln in scan.stdout.splitlines() if ln.startswith(marker)), None)
if marker_line is None:
    raise SystemExit('SAFE ACTION: STOP — SCAN DID NOT EMIT @@SCAN_JSON@@')
scan_info = json.loads(marker_line[len(marker):])
SAFE_ACTION = scan_info.get('SAFE_ACTION')
print('PARSED SAFE ACTION:', SAFE_ACTION)
if SAFE_ACTION not in {'START NEW CAMPAIGN', 'RESUME'}:
    raise SystemExit('Not safe to continue: ' + str(SAFE_ACTION))
SCAN_GATE_PASS = True
print('CELL 0 GATE: PASS')

In [ ]:
import subprocess, sys, torch, os

assert globals().get('SCAN_GATE_PASS') is True, 'Run Cell 0 successfully first.'
TEST_GATE_PASS = False
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU before running.'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)

paths = [
    'experiments/ARK-020-V4/ark020_v4_core.py',
    'experiments/ARK-020-V4/run_ark020_v4.py',
    'tests/test_ark020_v4.py',
    'experiments/ARK-019/ark019_v4_core.py',
    'experiments/ARK-019/run_ark019_v4.py',
    'experiments/ARK-019/run_ark019_v3.py',
    'experiments/ARK-018/ark018_v3_common.py',
    'experiments/ARK-018/ark018_v3_binding_fast.py',
]
for p in paths:
    subprocess.run([sys.executable, '-m', 'py_compile', os.path.join(REPO, p)], check=True)
print('compile gate: PASS on', len(paths), 'files')

print('Running V4 tests with LIVE verbose output...')
test_cmd = [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests',
            '-p', 'test_ark020_v4.py', '-v']
tests = subprocess.run(test_cmd, cwd=REPO)
if tests.returncode != 0:
    raise SystemExit(f'test suite failed (return code {tests.returncode}) — DO NOT RUN campaign')
TEST_GATE_PASS = True
print('CELL 1 GATE: PASS — all V4 tests passed')

In [ ]:
# Full campaign. Exact-resumable: rerun Cell 0, Cell 1, then this cell on a fresh T4.
import os, subprocess, sys, json
from pathlib import Path

assert globals().get('SCAN_GATE_PASS') is True, 'Cell 0 gate not passed.'
assert globals().get('TEST_GATE_PASS') is True, 'Cell 1 test gate not passed.'
runner = os.path.join(REPO, 'experiments/ARK-020-V4/run_ark020_v4.py')
root = Path('/content/drive/MyDrive/genisis-arkenstone/ARK020_V4_CONTINUAL')
env = dict(os.environ); env['PYTHONUNBUFFERED'] = '1'
print('=== STARTING / RESUMING ARK-020 V4 ===', flush=True)
proc = subprocess.run([sys.executable, runner, '--mode','all'], cwd=REPO, env=env)
print('CAMPAIGN RETURN CODE:', proc.returncode)

if proc.returncode != 0:
    failure = root / 'ARK-020_V4_FAILURE.json'
    if failure.exists():
        print('\n===== REAL V4 FAILURE RECEIPT =====')
        try:
            f = json.loads(failure.read_text())
            print('EXCEPTION:', f.get('exception'))
            print('MESSAGE:', f.get('message'))
            print(f.get('traceback', failure.read_text()))
        except Exception:
            print(failure.read_text())
    else:
        print('No failure receipt found at', failure)
    partial = root / 'ARKENSTONE_ARK020_V4_CONTINUAL_PARTIAL.zip'
    if partial.exists(): print('PARTIAL BUNDLE:', partial)
    raise SystemExit('ARK-020 V4 child process failed — traceback printed above')

result = root / 'ARK-020_V4_RESULT.json'
session = root / 'SESSION_STATE.json'
if result.exists():
    r = json.loads(result.read_text())
    print('SCIENTIFIC CAMPAIGN STATUS:', r.get('status'))
    print('VERDICT:', r.get('decision', {}).get('verdict', r.get('verdict')))
elif session.exists():
    s = json.loads(session.read_text())
    print('SESSION STATUS:', s.get('status'))
    print('MESSAGE:', s.get('message'))
    if s.get('status') == 'PARTIAL_SESSION':
        print('This is expected for a multi-session run. On the next T4: rerun Cell 0 → Cell 1 → Cell 2.')
else:
    print('Process returned 0 but no result/session receipt was found. Inspect the Drive root before rerunning.')

In [ ]:
from pathlib import Path
import json, hashlib

root = Path('/content/drive/MyDrive/genisis-arkenstone/ARK020_V4_CONTINUAL')
print('CAMPAIGN ROOT:', root)
for name in ['PREEXECUTION_GATE.json','EXACT_RESUME_SMOKE_V4.json','SESSION_STATE.json','ARK-020_V4_RESULT.json','ARK-020_V4_FAILURE.json']:
    p = root / name
    if p.exists():
        print('\n===', name, '===')
        text = p.read_text()
        print(text[:6000])

final_zip = root / 'ARKENSTONE_ARK020_V4_CONTINUAL_RESULTS.zip'
partial_zip = root / 'ARKENSTONE_ARK020_V4_CONTINUAL_PARTIAL.zip'
z = final_zip if final_zip.exists() else partial_zip if partial_zip.exists() else None
if z is not None:
    print('\nBUNDLE:', z, '|', z.stat().st_size, 'bytes')
    sha_path = Path(str(z) + '.sha256')
    if sha_path.exists():
        recorded = sha_path.read_text().strip()
        print('SHA256 SIDECAR:', recorded)
    print('Bundle remains in Drive; download manually when needed.')
else:
    print('No bundle yet.')